# 📊 데이터 분석 입문 - 3편: 데이터 전처리 (Data Cleaning)

> **목표**: 실제 데이터의 결측치, 이상치, 변환 문제를 처리하는 법을 배웁니다.
>
> **현실**: 데이터 분석 시간의 **70~80%**는 전처리에 쓰입니다. 이걸 잘해야 분석 결과를 믿을 수 있어요.
>
> **소요 시간**: 약 90분

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
!apt-get install -y fonts-nanum > /dev/null 2>&1
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 준비 완료!")

In [ ]:
# 일부러 "지저분한" 데이터를 만들겠습니다
np.random.seed(42)
n = 150

df = pd.DataFrame({
    '학생ID': [f'S{str(i).zfill(3)}' for i in range(1, n+1)],
    '성별': np.random.choice(['남', '여', '남 ', ' 여', None], n, p=[0.4,0.4,0.05,0.05,0.1]),
    '학년': np.random.choice([1, 2, 3, -1, 99], n, p=[0.3,0.35,0.25,0.05,0.05]),
    'ICT학교사용': np.where(np.random.random(n) < 0.1, np.nan, 
                     np.random.choice([1,2,3,4,5], n)),
    'ICT가정사용': np.where(np.random.random(n) < 0.08, np.nan,
                     np.random.choice([1,2,3,4,5], n)),
    '하루ICT시간': np.where(np.random.random(n) < 0.05, np.nan,
                     np.clip(np.random.normal(3, 1.5, n), 0, 15)),
    '수학점수': np.where(np.random.random(n) < 0.07, np.nan,
                    np.random.normal(500, 70, n)),
})
# 이상치 추가
df.loc[5, '수학점수'] = 950
df.loc[10, '하루ICT시간'] = 24
df.loc[15, '수학점수'] = -50

print(f"데이터 크기: {df.shape}")
df.head(10)

보시면 여러 문제가 보입니다:
- 성별에 공백이 섞여 있음 (`'남 '`, `' 여'`)
- 성별에 `None` (결측치)
- 학년에 `-1`, `99` 같은 이상한 값
- 수학점수에 `950`, `-50` 같은 불가능한 값
- 여러 열에 `NaN` (빈 값)

이런 걸 하나씩 정리해봅시다.

---
## 1. 결측치 (Missing Values) 처리

### 1.1 결측치 확인

In [ ]:
# 열별 결측치 수
print("=== 결측치 현황 ===")
print(df.isnull().sum())
print()
print("=== 결측치 비율 ===")
print((df.isnull().sum() / len(df) * 100).round(1).astype(str) + '%')

In [ ]:
# 시각화로 결측치 패턴 확인
fig, ax = plt.subplots(figsize=(10, 4))
결측 = df.isnull().sum()
결측 = 결측[결측 > 0].sort_values(ascending=False)
결측.plot(kind='bar', color='coral', edgecolor='white', ax=ax)
ax.set_title('변수별 결측치 수')
ax.set_ylabel('결측치 수')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 1.2 결측치 처리 방법

세 가지 전략이 있습니다:

| 방법 | 언제 쓰나? | 장점 | 단점 |
|------|-----------|------|------|
| **삭제** | 결측치가 적을 때 (5% 미만) | 간단 | 데이터 손실 |
| **평균/중앙값 대체** | 수치형, 결측이 랜덤일 때 | 데이터 보존 | 분산이 줄어듦 |
| **최빈값 대체** | 범주형 변수 | 데이터 보존 | 편향 가능 |

In [ ]:
# 방법 1: 삭제 — 특정 열에 결측이 있는 행 제거
df_삭제 = df.dropna(subset=['수학점수'])
print(f"삭제 전: {len(df)}행 → 삭제 후: {len(df_삭제)}행")

In [ ]:
# 방법 2: 평균으로 대체
df_대체 = df.copy()
df_대체['수학점수'] = df_대체['수학점수'].fillna(df_대체['수학점수'].mean())
df_대체['하루ICT시간'] = df_대체['하루ICT시간'].fillna(df_대체['하루ICT시간'].median())  # 중앙값

print(f"수학점수 결측 수: {df_대체['수학점수'].isnull().sum()}")  # 0이어야 함

In [ ]:
# 방법 3: 최빈값으로 대체 (범주형)
df_대체['성별'] = df_대체['성별'].fillna(df_대체['성별'].mode()[0])
print(f"성별 결측 수: {df_대체['성별'].isnull().sum()}")

### ✏️ 연습 3-1
`ICT학교사용`과 `ICT가정사용`의 결측치를 **중앙값**으로 대체하세요.

In [ ]:
# ✏️ 여기에 코드를 작성하세요
df_대체['ICT학교사용'] = df_대체['ICT학교사용'].fillna(___)
df_대체['ICT가정사용'] = df_대체['ICT가정사용'].fillna(___)

# 확인
print(df_대체[['ICT학교사용', 'ICT가정사용']].isnull().sum())

---
## 2. 이상치 (Outliers) 처리

### 2.1 이상치 탐지

In [ ]:
# 수학점수의 이상치 확인
print(df['수학점수'].describe())
print(f"\n최솟값: {df['수학점수'].min()}")
print(f"최댓값: {df['수학점수'].max()}")

In [ ]:
# 박스플롯으로 이상치 시각적 확인
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot(df['수학점수'].dropna())
axes[0].set_title('수학점수 (이상치 포함)')

axes[1].boxplot(df['하루ICT시간'].dropna())
axes[1].set_title('하루 ICT 사용시간 (이상치 포함)')

plt.tight_layout()
plt.show()

In [ ]:
# IQR 방법으로 이상치 탐지
def 이상치_탐지(series, 변수명=""):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    하한 = Q1 - 1.5 * IQR
    상한 = Q3 + 1.5 * IQR
    이상치 = series[(series < 하한) | (series > 상한)]
    print(f"📊 {변수명}: Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
    print(f"   정상 범위: [{하한:.1f}, {상한:.1f}]")
    print(f"   이상치 {len(이상치)}개: {이상치.values}")
    return 하한, 상한

하한_수학, 상한_수학 = 이상치_탐지(df['수학점수'].dropna(), "수학점수")
print()
하한_ict, 상한_ict = 이상치_탐지(df['하루ICT시간'].dropna(), "하루ICT시간")

In [ ]:
# 이상치 처리: 정상 범위로 클리핑 (제거 대신)
df_정리 = df_대체.copy()
df_정리['수학점수'] = df_정리['수학점수'].clip(lower=200, upper=800)
df_정리['하루ICT시간'] = df_정리['하루ICT시간'].clip(lower=0, upper=15)

print("처리 후:")
print(f"  수학점수 범위: {df_정리['수학점수'].min():.0f} ~ {df_정리['수학점수'].max():.0f}")
print(f"  ICT시간 범위: {df_정리['하루ICT시간'].min():.1f} ~ {df_정리['하루ICT시간'].max():.1f}")

---
## 3. 데이터 정제 (Cleaning)

### 3.1 문자열 정리

In [ ]:
# 성별의 문제 확인
print("정리 전:")
print(df_정리['성별'].value_counts())

In [ ]:
# 공백 제거 + 통일
df_정리['성별'] = df_정리['성별'].str.strip()  # 앞뒤 공백 제거
print("\n정리 후:")
print(df_정리['성별'].value_counts())

### 3.2 이상한 값 처리

In [ ]:
# 학년에 -1, 99 같은 값이 있음
print("학년 분포:")
print(df_정리['학년'].value_counts().sort_index())

In [ ]:
# 유효하지 않은 학년을 NaN으로 바꾸기
df_정리.loc[~df_정리['학년'].isin([1, 2, 3]), '학년'] = np.nan
print("\n정리 후:")
print(df_정리['학년'].value_counts().sort_index())
print(f"결측: {df_정리['학년'].isnull().sum()}개")

---
## 4. 새로운 변수 만들기 (Feature Engineering)

기존 데이터에서 **분석에 유용한 새 변수를 파생**하는 작업입니다.
이게 분석의 질을 결정하는 경우가 많아요.

In [ ]:
# 4.1 구간 나누기 — 연속형 → 범주형
df_정리['수학수준'] = pd.cut(df_정리['수학점수'], 
                        bins=[0, 420, 482, 553, 625, 800],
                        labels=['Level1이하', 'Level2', 'Level3', 'Level4', 'Level5이상'])
print(df_정리['수학수준'].value_counts().sort_index())

In [ ]:
# 4.2 ICT 종합 점수 만들기
df_정리['ICT종합'] = (df_정리['ICT학교사용'] + df_정리['ICT가정사용']) / 2
print(df_정리[['ICT학교사용', 'ICT가정사용', 'ICT종합']].head())

In [ ]:
# 4.3 ICT 사용 그룹 나누기 (저/중/고)
df_정리['ICT사용그룹'] = pd.cut(df_정리['ICT종합'],
                            bins=[0, 2, 3.5, 5],
                            labels=['저사용', '중사용', '고사용'])
print(df_정리['ICT사용그룹'].value_counts())

In [ ]:
# 4.4 새 변수로 바로 분석!
그룹별_평균 = df_정리.groupby('ICT사용그룹')['수학점수'].agg(['mean', 'count', 'std']).round(1)
그룹별_평균.columns = ['평균점수', '학생수', '표준편차']
그룹별_평균

### ✏️ 연습 3-2
`하루ICT시간`을 3개 그룹으로 나누는 새 변수를 만드세요:
- 2시간 미만: "소극적"
- 2~4시간: "보통"
- 4시간 이상: "적극적"

In [ ]:
# ✏️ 여기에 코드를 작성하세요
df_정리['ICT시간그룹'] = pd.cut(df_정리['하루ICT시간'],
                            bins=___,
                            labels=___)
print(df_정리['ICT시간그룹'].value_counts())

---
## 5. 전처리 파이프라인 정리

실제 프로젝트에서는 이 과정을 **하나의 함수로 정리**해두면 편합니다.

In [ ]:
def 데이터_전처리(df_원본):
    """
    데이터 전처리 파이프라인
    입력: 원본 DataFrame
    출력: 정리된 DataFrame
    """
    df = df_원본.copy()
    
    # 1. 문자열 정리
    if '성별' in df.columns:
        df['성별'] = df['성별'].str.strip()
    
    # 2. 이상한 값 → NaN
    if '학년' in df.columns:
        df.loc[~df['학년'].isin([1, 2, 3]), '학년'] = np.nan
    
    # 3. 결측치 처리
    수치_열 = df.select_dtypes(include=[np.number]).columns
    for 열 in 수치_열:
        df[열] = df[열].fillna(df[열].median())
    
    범주_열 = df.select_dtypes(include=['object']).columns
    for 열 in 범주_열:
        df[열] = df[열].fillna(df[열].mode()[0])
    
    # 4. 이상치 클리핑
    if '수학점수' in df.columns:
        df['수학점수'] = df['수학점수'].clip(200, 800)
    if '하루ICT시간' in df.columns:
        df['하루ICT시간'] = df['하루ICT시간'].clip(0, 15)
    
    # 5. 파생 변수
    if 'ICT학교사용' in df.columns and 'ICT가정사용' in df.columns:
        df['ICT종합'] = (df['ICT학교사용'] + df['ICT가정사용']) / 2
    
    return df

# 사용
df_최종 = 데이터_전처리(df)
print(f"✅ 전처리 완료! {df_최종.shape[0]}행, {df_최종.shape[1]}열")
print(f"   결측치: {df_최종.isnull().sum().sum()}개")

---
## 📝 3편 정리

| 문제 | 확인 방법 | 처리 방법 |
|------|----------|----------|
| 결측치 | `isnull().sum()` | 삭제 / 평균·중앙값 대체 |
| 이상치 | 박스플롯, IQR | 제거 / 클리핑 |
| 공백·오타 | `value_counts()` | `str.strip()`, `replace()` |
| 이상한 값 | 도메인 지식으로 판단 | NaN 변환 후 처리 |
| 파생 변수 | 분석 목적에 따라 | `pd.cut()`, 산술 연산 |

### 다음 편 예고
4편에서는 **통계 분석** (상관분석, t-test, 그룹 비교)을 배웁니다!